# ARFIMAX and NARX Rolling Forecasts

Builds the two exogenous models on the same rolling-window protocol as the rest of the project (train 1585, test 679, test window 2022-04-20 → 2024-12-31) and compares them head-to-head against their exog-free counterparts (ARFIMA, NAR) and the strongest benchmarks (HAR, LSTM).

* **ARFIMAX(0, d, 1)** — two-stage: OLS on the 7 macro features, then ARFIMA on the residual. Refit **every step** (cheap: ≈ 0.1 s/fit).
* **NARX** — 7 RV lags + 7 macro features → 7 sigmoid hidden → linear out. Refit **every 22 days** (matches NAR / LSTM).

Exogenous matrix is the 1-day-lagged, RV-aligned macro panel from notebook 13 (no look-ahead). All seeds = 42.

In [1]:
from __future__ import annotations
import sys, time, warnings, logging
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
logging.getLogger('matplotlib').setLevel(logging.WARNING)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.econometric_models import ARFIMAXForecaster  # noqa: E402
from src.neural_models import NARXForecaster  # noqa: E402
from src.forecast_engine import rolling_forecast_exog  # noqa: E402
from src.metrics import mse, qlike  # noqa: E402

PROCESSED = PROJECT_ROOT / 'data' / 'processed'
TABLES = PROJECT_ROOT / 'results' / 'tables'
TICKERS = ['AAPL', 'AMZN', 'JPM']
TRAIN_WINDOW, TEST_SIZE = 1585, 679

# Lagged, RV-aligned macro (built in notebook 13; rebuild here if absent).
macro = pd.read_csv(PROCESSED / 'macro_features.csv', parse_dates=['date']).set_index('date')
macro_lagged = macro.shift(1).dropna()

In [2]:
exog_forecasts = {}
for tick in TICKERS:
    rv = pd.read_parquet(PROCESSED / f'{tick}_daily_rv.parquet')['log_rv']
    X = macro_lagged.reindex(rv.index, method='ffill')
    assert len(X) == len(rv) == 2264 and X.isna().sum().sum() == 0

    # ARFIMAX — refit every step
    ax_cache = PROCESSED / f'forecasts_arfimax_{tick}.csv'
    if ax_cache.exists():
        ax_out = pd.read_csv(ax_cache, parse_dates=['date']).set_index('date')
    else:
        t0 = time.time()
        ax_out = rolling_forecast_exog(lambda: ARFIMAXForecaster(q=1), rv, X,
                                       train_window=TRAIN_WINDOW, test_size=TEST_SIZE,
                                       refit_every=1, desc=f'{tick}-ARFIMAX')
        ax_out = ax_out.rename(columns={'forecast': 'arfimax'})[['actual', 'arfimax']]
        ax_out.index.name = 'date'
        ax_out.to_csv(ax_cache, float_format='%.6f')
        print(f'  {tick} ARFIMAX: {time.time()-t0:.1f}s -> {ax_cache.name}')

    # NARX — refit every 22 days
    nx_cache = PROCESSED / f'forecasts_narx_{tick}.csv'
    if nx_cache.exists():
        nx_out = pd.read_csv(nx_cache, parse_dates=['date']).set_index('date')
    else:
        t0 = time.time()
        nx_out = rolling_forecast_exog(lambda: NARXForecaster(n_lags=7, hidden_units=7, seed=42), rv, X,
                                       train_window=TRAIN_WINDOW, test_size=TEST_SIZE,
                                       refit_every=22, desc=f'{tick}-NARX')
        nx_out = nx_out.rename(columns={'forecast': 'narx'})[['actual', 'narx']]
        nx_out.index.name = 'date'
        nx_out.to_csv(nx_cache, float_format='%.6f')
        print(f'  {tick} NARX: {time.time()-t0:.1f}s -> {nx_cache.name}')

    exog_forecasts[tick] = {'arfimax': ax_out, 'narx': nx_out}
print('done')

AAPL-ARFIMAX:   0%|          | 0/679 [00:00<?, ?it/s]

  AAPL ARFIMAX: 49.6s -> forecasts_arfimax_AAPL.csv


AAPL-NARX:   0%|          | 0/679 [00:00<?, ?it/s]

  AAPL NARX: 10.7s -> forecasts_narx_AAPL.csv


AMZN-ARFIMAX:   0%|          | 0/679 [00:00<?, ?it/s]

...

  AMZN NARX: 13.8s -> forecasts_narx_AMZN.csv


JPM-ARFIMAX:   0%|          | 0/679 [00:00<?, ?it/s]

  JPM ARFIMAX: 59.4s -> forecasts_arfimax_JPM.csv


JPM-NARX:   0%|          | 0/679 [00:00<?, ?it/s]

  JPM NARX: 13.9s -> forecasts_narx_JPM.csv
done


In [3]:
# Compare ARFIMA / ARFIMAX / NAR / NARX / HAR / LSTM
old = pd.read_csv(PROCESSED / 'all_forecasts_AAPL.csv').columns  # sanity for col names
rows = []
compare_models = ['ARFIMA', 'ARFIMAX', 'NAR', 'NARX', 'HAR', 'LSTM']
base_col = {'ARFIMA': 'arfima', 'NAR': 'nar', 'HAR': 'har', 'LSTM': 'lstm'}
for name in compare_models:
    row = {'Model': name}
    mses, qlikes = [], []
    for tick in TICKERS:
        if name in ('ARFIMA', 'NAR', 'HAR', 'LSTM'):
            df = pd.read_csv(PROCESSED / f'all_forecasts_{tick}.csv')
            a, f = df['actual'].values, df[base_col[name]].values
        elif name == 'ARFIMAX':
            d = exog_forecasts[tick]['arfimax']; a, f = d['actual'].values, d['arfimax'].values
        else:  # NARX
            d = exog_forecasts[tick]['narx']; a, f = d['actual'].values, d['narx'].values
        m, q = mse(a, f), qlike(a, f)
        row[f'{tick}_MSE'] = m; row[f'{tick}_QLIKE'] = q
        mses.append(m); qlikes.append(q)
    row['Avg_MSE'] = float(np.mean(mses)); row['Avg_QLIKE'] = float(np.mean(qlikes))
    rows.append(row)
res = pd.DataFrame(rows).set_index('Model')
res.to_csv(TABLES / '13_arfimax_narx_results.csv', float_format='%.4f')
print('saved 13_arfimax_narx_results.csv')
print(res.round(4).to_string())
print('\n--- exog vs exog-free deltas (Avg MSE) ---')
print(f"  ARFIMAX - ARFIMA = {res.loc['ARFIMAX','Avg_MSE'] - res.loc['ARFIMA','Avg_MSE']:+.4f}")
print(f"  NARX    - NAR    = {res.loc['NARX','Avg_MSE'] - res.loc['NAR','Avg_MSE']:+.4f}")

saved 13_arfimax_narx_results.csv
         AAPL_MSE  AAPL_QLIKE  AMZN_MSE  AMZN_QLIKE  JPM_MSE  JPM_QLIKE  Avg_MSE  Avg_QLIKE
Model                                                                                      
ARFIMA     0.0615      0.1433    0.0523      0.1160   0.0570     0.1394   0.0569     0.1329
ARFIMAX    0.0589      0.1363    0.0537      0.1085   0.0590     0.1453   0.0572     0.1300
NAR        0.0621      0.1448    0.0566      0.1280   0.0592     0.1458   0.0593     0.1395
NARX       0.0632      0.1420    0.0539      0.1151   0.0565     0.1399   0.0579     0.1323
HAR        0.0627      0.1466    0.0526      0.1181   0.0579     0.1423   0.0578     0.1357
LSTM       0.0629      0.1452    0.0558      0.1256   0.0593     0.1436   0.0594     0.1381

--- exog vs exog-free deltas (Avg MSE) ---
  ARFIMAX - ARFIMA = +0.0002
  NARX    - NAR    = -0.0015


## Findings

**Headline: the exogenous variables help — modestly for ARFIMAX, more clearly for NARX — but do not overturn ARFIMA's MSE lead.**

| Model | Avg MSE | Avg QLIKE |
| --- | ---: | ---: |
| ARFIMA | **0.0569** | 0.1329 |
| ARFIMAX | 0.0572 | **0.1300** |
| HAR | 0.0578 | 0.1357 |
| NARX | 0.0579 | 0.1323 |
| NAR | 0.0593 | 0.1395 |
| LSTM | 0.0594 | 0.1381 |

**1. ARFIMAX vs ARFIMA — a wash on MSE, a win on QLIKE.** Adding the 7 macro features leaves average MSE essentially unchanged (0.0572 vs 0.0569, +0.4 %) but *lowers* average QLIKE from 0.1329 to **0.1300** — the best QLIKE of any model in the project. The per-ticker picture is split: on **AAPL** ARFIMAX clearly beats ARFIMA on both metrics (MSE 0.0589 vs 0.0615), while on AMZN and JPM it is marginally worse on MSE. The two-stage OLS pulls the VIX / MKT / DEF signal out of the conditional mean, which helps most where the macro state is most informative (AAPL).

**2. NARX vs NAR — exogenous variables genuinely improve the neural model.** This is the cleanest exog result: NARX cuts NAR's average MSE by 0.0015 (0.0593 → 0.0579, −2.5 %) and QLIKE by 0.0072 (0.1395 → 0.1323, −5.2 %). On **JPM** NARX is the single best MSE model in this comparison (0.0565, beating even ARFIMA's 0.0570). This matches Bucci's central claim — the gains from neural architectures come precisely from nonlinear interactions with the exogenous predictors, not from the lagged RV alone.

**3. Does any exog model beat ARFIMA?** On **average MSE**, no — ARFIMA (0.0569) is still lowest, with ARFIMAX (0.0572) and NARX (0.0579) just behind. On **average QLIKE**, yes — ARFIMAX (0.1300) and NARX (0.1323) both beat ARFIMA (0.1329). And per-ticker, ARFIMAX wins AAPL and NARX wins JPM. The exogenous variables close most of the residual gap; whether they *overturn* the verdict is now a question for the formal MCS / encompassing tests in notebook 15.

**4. Honest note on fidelity.** This is NARX with 7 of Bucci's 11 predictors (no high-yield OAS, no two further series). The improvement direction is exactly Bucci's; the magnitude is smaller, consistent with the reduced predictor set and the very strong long memory of daily US-equity log-RV that already makes the linear ARFIMA hard to beat.